# Step 1 (RunPod): SMILES to molecular graphsSelf-contained notebook. It does not import from any repository and has nodependency on the GMACORE-TM package. Upload this single file to the pod andrun it top to bottom.Featurization is run once here. All three pretraining variants (ManualAug-CL,GMACORE, GMACORE-TM) then load the identical graphs, which is what makes thecomparison between them a clean ablation rather than three runs on slightlydifferent data.**This step is CPU-bound.** RDKit parsing does not use the GPU. What mattershere is vCPU count and RAM, not VRAM. The GPU is used from step 2 onward; theenvironment check below verifies it is present and working so that a brokeninstall is found now rather than after this step.## Before running: persistent storageRunPod container disk is wiped when the pod stops. Write the output to anetwork volume mounted at `/workspace`, otherwise the featurized corpus is lostand has to be rebuilt. The default paths below assume `/workspace` exists.

In [ ]:
# Silence warnings as requested, before anything else imports.
import warnings

warnings.filterwarnings("ignore")

import os

os.environ["PYTHONWARNINGS"] = "ignore"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import logging
import sys

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
print(f"Python {sys.version.split()[0]}")

## Install dependenciesThe RunPod PyTorch template already provides `torch`. Two packages are added:- `rdkit` for SMILES parsing- `torch-geometric` for the graph data structuresOnly the pure-Python part of PyTorch Geometric is needed. This code uses`GCNConv`, `global_mean_pool` and the `DataLoader`, none of which require thecompiled extensions (`pyg_lib`, `torch_scatter`, `torch_sparse`). Skipping themavoids the usual version-matching problems, at a small cost in message-passingspeed that does not matter for a 3-layer GCN.Run this cell once per pod.

In [ ]:
import subprocess


def pip_install(*packages):
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *packages],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
        raise RuntimeError(f"Failed to install {packages}")
    print(f"installed: {' '.join(packages)}")


missing = []
for module, package in [
    ("rdkit", "rdkit"),
    ("torch_geometric", "torch-geometric"),
    ("tqdm", "tqdm"),
]:
    try:
        __import__(module)
        print(f"present  : {module}")
    except ImportError:
        missing.append(package)

if missing:
    pip_install(*missing)
else:
    print("\nAll dependencies already available.")

## Environment checkConfirms the GPU is visible and that PyTorch can actually allocate on it. Apod that reports a GPU but fails to allocate is a broken image, and it is muchcheaper to discover that now.

In [ ]:
import torch

print(f"PyTorch : {torch.__version__}")

import rdkit
import torch_geometric

print(f"PyG     : {torch_geometric.__version__}")
print(f"RDKit   : {rdkit.__version__}")
print(f"vCPUs   : {os.cpu_count()}")

try:
    import psutil

    print(f"RAM     : {psutil.virtual_memory().total / 1024**3:.0f} GB")
except ImportError:
    pass

if torch.cuda.is_available():
    print(f"\nCUDA    : {torch.version.cuda}")
    for i in range(torch.cuda.device_count()):
        properties = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {properties.name} ({properties.total_memory / 1024**3:.0f} GB)")

    # Prove the device actually works rather than merely being listed.
    probe = torch.randn(1000, 1000, device="cuda")
    _ = (probe @ probe).sum().item()
    del probe
    torch.cuda.empty_cache()
    print("  allocation test: OK")
else:
    print("\nCUDA    : NOT AVAILABLE")
    print("  Fine for this step, but step 2 needs a GPU. Check the pod template.")

## ConfigurationSet `SMILES_FILE` to your corpus. Start with a small `LIMIT` to confirm thepipeline runs end to end, then set it to `None` and rerun for the full corpus.

In [ ]:
from pathlib import Path

# ----- paths (on the persistent volume) -----
WORKSPACE = Path("/workspace") if Path("/workspace").exists() else Path.cwd()
SMILES_FILE = WORKSPACE / "data" / "pubchem" / "pubchem_10m_clean.txt"
OUTPUT_DIR = WORKSPACE / "data" / "graphs" / "pubchem"

# ----- corpus size -----
LIMIT = 100_000          # set to None for the whole file
SHARD_SIZE = 50_000      # graphs per shard file

# ----- featurization (defaults match the manuscript: 2D, heavy atoms only) -----
ADD_EXPLICIT_HYDROGENS = False
GENERATE_CONFORMER = False       # leave False; conformers are slow and unused
MIN_HEAVY_ATOMS = 2
MAX_HEAVY_ATOMS = None
KEEP_LARGEST_FRAGMENT = True     # strip counterions from salts such as "CCO.Cl"

# ----- parallelism -----
NUM_WORKERS = max(1, (os.cpu_count() or 2) - 1)
CHUNK_SIZE = 500

OVERWRITE = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Workspace : {WORKSPACE}")
print(f"Input     : {SMILES_FILE}")
print(f"Output    : {OUTPUT_DIR}")
print(f"Limit     : {LIMIT if LIMIT is not None else 'no limit'}")
print(f"Workers   : {NUM_WORKERS}")

if WORKSPACE == Path.cwd():
    print("\nWARNING: /workspace not found, writing to the container disk.")
    print("Output will be LOST when the pod stops. Attach a network volume.")

## FeaturizerDefined inline so this notebook stands alone. The schema matches the manuscript:| Field | Shape | Contents || --- | --- | --- || `x_cat` | `[atoms, 2]` | atom identity, chirality tag || `x_phys` | `[atoms, 8]` | weight, logP, charge, hybridization, aromaticity, H count, valence, degree || `edge_index` | `[2, 2 x bonds]` | bonds, both directions || `edge_attr` | `[2 x bonds, 5]` | bond type, direction, conjugation, rotatable, length |The two node groups are concatenated before encoding; the split is notationaland mirrors the paper's description of heterogeneous node features.Featurization is two dimensional by default, so the bond-length channel iszero. That matches the paper's Limitations section, which states threedimensional conformational information is not incorporated.

In [ ]:
from dataclasses import dataclass
from typing import List, Optional, Tuple

import torch
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, rdMolTransforms
from torch_geometric.data import Data

RDLogger.DisableLog("rdApp.*")

NUM_NODE_CATEGORICAL = 2
NUM_NODE_PHYSICOCHEMICAL = 8
NUM_EDGE_FEATURES = 5
NODE_DIM = NUM_NODE_CATEGORICAL + NUM_NODE_PHYSICOCHEMICAL
EDGE_DIM = NUM_EDGE_FEATURES

ATOM_LIST = list(range(1, 119))
CHIRALITY_LIST = [
    Chem.rdchem.ChiralType.CHI_UNSPECIFIED,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW,
    Chem.rdchem.ChiralType.CHI_OTHER,
]
BOND_LIST = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC,
]
BOND_DIR_LIST = [
    Chem.rdchem.BondDir.NONE,
    Chem.rdchem.BondDir.ENDUPRIGHT,
    Chem.rdchem.BondDir.ENDDOWNRIGHT,
]


@dataclass
class FeaturizerConfig:
    add_explicit_hydrogens: bool = False
    generate_conformer: bool = False
    min_heavy_atoms: int = 2
    max_heavy_atoms: Optional[int] = None
    keep_largest_fragment: bool = True


class MolecularFeaturizer:
    """Builds a PyG Data object from a SMILES string."""

    def __init__(self, config: Optional[FeaturizerConfig] = None) -> None:
        self.config = config or FeaturizerConfig()
        self._atom_index = {z: i for i, z in enumerate(ATOM_LIST)}
        self._chirality_index = {c: i for i, c in enumerate(CHIRALITY_LIST)}
        self._bond_index = {b: i for i, b in enumerate(BOND_LIST)}
        self._bond_dir_index = {d: i for i, d in enumerate(BOND_DIR_LIST)}
        # Element descriptors depend only on the symbol, so they are cached.
        self._element_cache = {}

    def _element_contributions(self, symbol: str) -> Tuple[float, float]:
        if symbol in self._element_cache:
            return self._element_cache[symbol]

        weight, logp = 0.0, 0.0
        probe = Chem.MolFromSmiles(f"[{symbol}]")
        if probe is not None:
            try:
                weight = float(Descriptors.ExactMolWt(probe))
                logp = float(Descriptors.MolLogP(probe))
            except Exception:
                pass

        self._element_cache[symbol] = (weight, logp)
        return weight, logp

    def _node_features(self, mol) -> Tuple[torch.Tensor, torch.Tensor]:
        categorical, physicochemical = [], []
        for atom in mol.GetAtoms():
            categorical.append([
                self._atom_index.get(atom.GetAtomicNum(), 0),
                self._chirality_index.get(atom.GetChiralTag(), 0),
            ])
            weight, logp = self._element_contributions(atom.GetSymbol())
            physicochemical.append([
                weight,
                logp,
                float(atom.GetFormalCharge()),
                float(int(atom.GetHybridization())),
                float(atom.GetIsAromatic()),
                float(atom.GetTotalNumHs()),
                float(atom.GetTotalValence()),
                float(atom.GetDegree()),
            ])
        return (
            torch.tensor(categorical, dtype=torch.long),
            torch.tensor(physicochemical, dtype=torch.float),
        )

    @staticmethod
    def _is_rotatable(bond) -> bool:
        return (
            bond.GetBondType() == Chem.rdchem.BondType.SINGLE
            and not bond.IsInRing()
            and bond.GetBeginAtom().GetDegree() > 1
            and bond.GetEndAtom().GetDegree() > 1
        )

    @staticmethod
    def _bond_length(mol, start: int, end: int) -> float:
        if mol.GetNumConformers() == 0:
            return 0.0
        conformer = mol.GetConformer()
        if not conformer.Is3D():
            return 0.0
        try:
            return float(rdMolTransforms.GetBondLength(conformer, start, end))
        except Exception:
            return 0.0

    def _edge_features(self, mol) -> Tuple[torch.Tensor, torch.Tensor]:
        rows: List[int] = []
        cols: List[int] = []
        features: List[List[float]] = []

        for bond in mol.GetBonds():
            start, end = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
            feature = [
                float(self._bond_index.get(bond.GetBondType(), 0)),
                float(self._bond_dir_index.get(bond.GetBondDir(), 0)),
                float(bond.GetIsConjugated()),
                float(self._is_rotatable(bond)),
                self._bond_length(mol, start, end),
            ]
            rows += [start, end]
            cols += [end, start]
            features += [feature, feature]

        if not rows:
            return (
                torch.empty((2, 0), dtype=torch.long),
                torch.empty((0, NUM_EDGE_FEATURES), dtype=torch.float),
            )
        return (
            torch.tensor([rows, cols], dtype=torch.long),
            torch.tensor(features, dtype=torch.float),
        )

    def __call__(self, smiles: str, label=None) -> Optional[Data]:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        # PubChem contains many salts and mixtures written as dot-separated
        # fragments, for example "CCO.Cl". Left alone these become a single
        # disconnected graph, and mean pooling would blend the counterion into
        # the molecule's representation. Keeping only the largest fragment
        # yields the parent structure, which is what the property labels refer
        # to in the downstream benchmarks.
        if self.config.keep_largest_fragment and "." in smiles:
            fragments = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=False)
            if len(fragments) > 1:
                mol = max(fragments, key=lambda fragment: fragment.GetNumHeavyAtoms())

        if self.config.add_explicit_hydrogens:
            mol = Chem.AddHs(mol)

        try:
            Chem.SanitizeMol(mol)
        except Exception:
            return None

        if self.config.generate_conformer:
            if AllChem.EmbedMolecule(mol, AllChem.ETKDG()) != 0:
                return None
            if AllChem.MMFFOptimizeMolecule(mol) != 0:
                AllChem.UFFOptimizeMolecule(mol)

        if mol.GetNumAtoms() == 0:
            return None

        heavy = mol.GetNumHeavyAtoms()
        if heavy < self.config.min_heavy_atoms:
            return None
        if self.config.max_heavy_atoms is not None and heavy > self.config.max_heavy_atoms:
            return None

        x_cat, x_phys = self._node_features(mol)
        edge_index, edge_attr = self._edge_features(mol)

        # An isolated atom carries no message-passing signal.
        if edge_index.size(1) == 0:
            return None

        data = Data(
            x_cat=x_cat,
            x_phys=x_phys,
            edge_index=edge_index,
            edge_attr=edge_attr,
            num_nodes=x_cat.size(0),
        )
        data.smiles = smiles
        if label is not None:
            data.y = torch.tensor([label], dtype=torch.float)
        return data


print(f"Featurizer defined. node_dim={NODE_DIM}, edge_dim={EDGE_DIM}")

## Shard storageGraphs are written in numbered shards with a manifest, rather than one largefile. This keeps peak memory at one shard instead of the whole corpus, lets aninterrupted run keep what it already finished, and lets a later run load asubset without re-featurizing.

In [ ]:
import json

MANIFEST_NAME = "manifest.json"


def shard_path(directory, index):
    return os.path.join(str(directory), f"shard_{index:05d}.pt")


def write_shard(directory, index, graphs):
    os.makedirs(str(directory), exist_ok=True)
    path = shard_path(directory, index)
    torch.save(list(graphs), path)
    return path


def write_manifest(directory, **fields):
    path = os.path.join(str(directory), MANIFEST_NAME)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(fields, handle, indent=2)
    return path


def read_manifest(directory):
    with open(os.path.join(str(directory), MANIFEST_NAME), encoding="utf-8") as handle:
        return json.load(handle)


class ShardedGraphDataset(torch.utils.data.Dataset):
    """Dataset backed by graph shards on disk.

    With lazy=False every shard is read into memory once, which is fastest when
    the corpus fits in RAM. With lazy=True one shard is held at a time, which
    keeps memory flat for corpora that do not fit.
    """

    def __init__(self, directory, max_shards=None, lazy=False):
        self.directory = str(directory)
        self.lazy = lazy
        self.manifest = read_manifest(directory)

        counts = list(self.manifest["shard_counts"])
        if max_shards is not None:
            counts = counts[:max_shards]
        self.shard_counts = counts
        self.num_shards = len(counts)

        self.offsets = [0]
        for count in counts:
            self.offsets.append(self.offsets[-1] + count)

        self._cached_index = None
        self._cached_shard = None
        self._graphs = None

        if not lazy:
            self._graphs = []
            for index in range(self.num_shards):
                self._graphs.extend(
                    torch.load(shard_path(self.directory, index), weights_only=False)
                )

    def __len__(self):
        return len(self._graphs) if self._graphs is not None else self.offsets[-1]

    def _locate(self, index):
        low, high = 0, self.num_shards - 1
        while low < high:
            middle = (low + high) // 2
            if index < self.offsets[middle + 1]:
                high = middle
            else:
                low = middle + 1
        return low, index - self.offsets[low]

    def __getitem__(self, index):
        if self._graphs is not None:
            return self._graphs[index]
        shard_index, position = self._locate(index)
        if shard_index != self._cached_index:
            self._cached_shard = torch.load(
                shard_path(self.directory, shard_index), weights_only=False
            )
            self._cached_index = shard_index
        return self._cached_shard[position]


print("Shard utilities defined.")

## Read the SMILES fileIf you do not yet have a corpus on the pod, the next cell will create a smallsample file so the pipeline can be exercised end to end. Replace it with thereal PubChem corpus before the production run.

In [ ]:
def read_smiles_file(path, limit=None):
    """Read newline-delimited SMILES, taking the first column of each line."""
    values = []
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            value = line.strip()
            if not value or value.startswith("#"):
                continue
            values.append(value.split()[0].split(",")[0])
            if limit is not None and len(values) >= limit:
                break
    return values


if not SMILES_FILE.exists():
    print(f"{SMILES_FILE} not found.")
    print("Creating a small sample file so the pipeline can be tested.\n")

    SAMPLE = [
        "CCO", "CC(=O)Oc1ccccc1C(=O)O", "CN1C=NC2=C1C(=O)N(C)C(=O)N2C",
        "c1ccccc1", "CC(C)Cc1ccc(cc1)C(C)C(=O)O", "CCN(CC)CC",
        "C1CCCCC1", "CC(=O)Nc1ccc(O)cc1", "OC(=O)c1ccccc1O",
        "CN1CCC[C@H]1c1cccnc1", "CCOC(=O)c1ccccc1", "Clc1ccccc1",
        "CC(C)(C)c1ccc(O)cc1", "NCCc1ccc(O)cc1", "CC1=CC(=O)CC(C)(C)C1",
    ]
    SMILES_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(SMILES_FILE, "w", encoding="utf-8") as handle:
        for i in range(400):
            handle.write(SAMPLE[i % len(SAMPLE)] + "\n")
    print(f"Wrote {SMILES_FILE} (400 lines, {len(SAMPLE)} unique)")

print(f"\nFile size: {SMILES_FILE.stat().st_size / 1024**2:,.2f} MB")
print("First 5 lines:")
with open(SMILES_FILE, encoding="utf-8") as handle:
    for i, line in enumerate(handle):
        if i >= 5:
            break
        print(f"  {line.rstrip()}")

In [ ]:
import time

start = time.perf_counter()
smiles_list = read_smiles_file(str(SMILES_FILE), limit=LIMIT)
print(f"Read {len(smiles_list):,} SMILES in {time.perf_counter() - start:.1f}s")

# Duplicates waste featurization time and fill the contrastive queue with
# identical negatives, so they are dropped while preserving order.
unique = list(dict.fromkeys(smiles_list))
if len(unique) != len(smiles_list):
    print(f"Removed {len(smiles_list) - len(unique):,} duplicates")
smiles_list = unique

print(f"Featurizing {len(smiles_list):,} unique molecules")

## FeaturizeRuns across all available cores. Each shard is written as soon as it fills, somemory stays bounded and an interrupted run keeps its completed shards.Molecules RDKit cannot parse or sanitize, and single atoms with no bonds, areskipped and counted. A few percent on PubChem is normal.

In [ ]:
from multiprocessing import Pool

from tqdm.auto import tqdm

FEATURIZER_CONFIG = FeaturizerConfig(
    add_explicit_hydrogens=ADD_EXPLICIT_HYDROGENS,
    generate_conformer=GENERATE_CONFORMER,
    min_heavy_atoms=MIN_HEAVY_ATOMS,
    max_heavy_atoms=MAX_HEAVY_ATOMS,
    keep_largest_fragment=KEEP_LARGEST_FRAGMENT,
)

# One featurizer per worker, built once, so the element cache survives across
# molecules handled by that worker.
_WORKER = None


def _init_worker(config):
    global _WORKER
    warnings.filterwarnings("ignore")
    RDLogger.DisableLog("rdApp.*")
    _WORKER = MolecularFeaturizer(config)


def _featurize_one(smiles):
    try:
        return _WORKER(smiles)
    except Exception:
        # One malformed entry must not kill the pool.
        return None

In [ ]:
manifest_file = OUTPUT_DIR / MANIFEST_NAME

if manifest_file.exists() and not OVERWRITE:
    print(f"Shards already exist at {OUTPUT_DIR}")
    print("Set OVERWRITE = True to rebuild, or continue to verification.")
else:
    start = time.perf_counter()
    buffer, shard_counts, shard_index = [], [], 0
    num_ok = num_failed = 0

    with Pool(NUM_WORKERS, initializer=_init_worker, initargs=(FEATURIZER_CONFIG,)) as pool:
        stream = pool.imap(_featurize_one, smiles_list, chunksize=CHUNK_SIZE)
        for graph in tqdm(stream, total=len(smiles_list), desc="Featurizing", unit="mol"):
            if graph is None:
                num_failed += 1
                continue
            buffer.append(graph)
            num_ok += 1
            if len(buffer) >= SHARD_SIZE:
                write_shard(OUTPUT_DIR, shard_index, buffer)
                shard_counts.append(len(buffer))
                shard_index += 1
                buffer = []

    if buffer:
        write_shard(OUTPUT_DIR, shard_index, buffer)
        shard_counts.append(len(buffer))

    elapsed = time.perf_counter() - start

    write_manifest(
        OUTPUT_DIR,
        num_graphs=num_ok,
        num_shards=len(shard_counts),
        shard_size=SHARD_SIZE,
        node_dim=NODE_DIM,
        edge_dim=EDGE_DIM,
        source_file=str(SMILES_FILE),
        num_requested=len(smiles_list),
        num_failed=num_failed,
        featurizer=FEATURIZER_CONFIG.__dict__,
        shard_counts=shard_counts,
    )

    print(f"\nFeaturized : {num_ok:,}")
    print(f"Skipped    : {num_failed:,} ({100 * num_failed / max(len(smiles_list), 1):.2f}%)")
    print(f"Shards     : {len(shard_counts)}")
    print(f"Elapsed    : {elapsed / 60:.1f} min ({num_ok / max(elapsed, 1e-9):,.0f} mol/s)")
    print(f"Written to : {OUTPUT_DIR}")

## VerifyLoads the shards back through the same class step 2 will use, and batches themthrough a real `DataLoader`. A shape problem fails here rather than an hourinto pretraining.

In [ ]:
dataset = ShardedGraphDataset(OUTPUT_DIR, lazy=False)

print(f"Graphs   : {len(dataset):,}")
print(f"Shards   : {dataset.num_shards}")
print(f"node_dim : {dataset.manifest['node_dim']}")
print(f"edge_dim : {dataset.manifest['edge_dim']}")

total_mb = sum(
    os.path.getsize(shard_path(OUTPUT_DIR, i)) for i in range(dataset.num_shards)
) / 1024**2
print(f"On disk  : {total_mb:,.1f} MB")

sample = dataset[0]
print("\nFirst graph:")
print(f"  SMILES     : {sample.smiles}")
print(f"  x_cat      : {tuple(sample.x_cat.shape)}")
print(f"  x_phys     : {tuple(sample.x_phys.shape)}")
print(f"  edge_index : {tuple(sample.edge_index.shape)}")
print(f"  edge_attr  : {tuple(sample.edge_attr.shape)}")

In [ ]:
import random

import numpy as np

random.seed(0)
indices = random.sample(range(len(dataset)), min(2000, len(dataset)))

atoms, bonds, problems = [], [], []
for i in indices:
    graph = dataset[i]
    n = graph.x_cat.shape[0]
    atoms.append(n)
    bonds.append(graph.edge_index.shape[1] // 2)

    if graph.edge_index.numel() and graph.edge_index.max() >= n:
        problems.append((i, "edge_index references a non-existent atom"))
    if graph.edge_index.shape[1] % 2 != 0:
        problems.append((i, "edges not stored symmetrically"))
    if graph.edge_index.shape[1] != graph.edge_attr.shape[0]:
        problems.append((i, "edge_index and edge_attr length disagree"))
    if not torch.isfinite(graph.x_phys).all():
        problems.append((i, "non-finite physicochemical feature"))

atoms, bonds = np.array(atoms), np.array(bonds)
print(f"Checked {len(indices):,} graphs\n")
print(f"Atoms : mean {atoms.mean():.1f}, median {np.median(atoms):.0f}, "
      f"range {atoms.min()}-{atoms.max()}")
print(f"Bonds : mean {bonds.mean():.1f}, range {bonds.min()}-{bonds.max()}")

if not GENERATE_CONFORMER:
    lengths = torch.cat([dataset[i].edge_attr[:, 4] for i in indices[:200]])
    print(f"\nBond-length channel max: {lengths.abs().max():.4f} (expected 0.0 for 2D)")

print(f"\n{'FAILED: ' + str(len(problems)) + ' problems' if problems else 'All structural checks passed.'}")
for index, message in problems[:10]:
    print(f"  graph {index}: {message}")

In [ ]:
from torch_geometric.loader import DataLoader

loader = DataLoader(dataset, batch_size=32, shuffle=True)
batch = next(iter(loader))

print(f"graphs     : {batch.num_graphs}")
print(f"x_cat      : {tuple(batch.x_cat.shape)}")
print(f"x_phys     : {tuple(batch.x_phys.shape)}")
print(f"edge_index : {tuple(batch.edge_index.shape)}")
print(f"edge_attr  : {tuple(batch.edge_attr.shape)}")
print(f"batch      : {tuple(batch.batch.shape)}")

assert batch.x_cat.shape[1] + batch.x_phys.shape[1] == dataset.manifest["node_dim"]
assert batch.edge_attr.shape[1] == dataset.manifest["edge_dim"]

# Confirm a batch moves to the GPU, since step 2 depends on this.
if torch.cuda.is_available():
    batch = batch.to("cuda")
    print(f"\nMoved to GPU: {batch.x_phys.device}")
    torch.cuda.empty_cache()

print("\nCorpus ready for pretraining.")

## Next step`OUTPUT_DIR` now holds the featurized corpus and is reused by all threepretraining runs without re-running RDKit.Report the graph count and skip rate before moving on. A skip rate well above afew percent points at a problem with the input file, which is far cheaper tofix now than after a pretraining run.Keep the pod's network volume; step 2 reads these shards from the same path.